# 01 — Exploratory Data Analysis
**Dynamic Pricing Engine** | Mohit | github.com/dswithmohit/dynamic-pricing-engine

---
### Goals
- Download the Kaggle dataset (or load cached synthetic version)
- Understand price/demand distributions across product categories
- Identify seasonality, outliers, and missing data
- Motivate the feature engineering choices in `02_feature_engineering.ipynb`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('Blues_d')

from src.data_loader import download_kaggle_dataset, load_raw, build_synthetic_dataset, eda_summary

## 1. Load Data

In [ ]:
# Download from Kaggle if not already present
# Requires kaggle.json in ~/.kaggle/  (chmod 600)
try:
    download_kaggle_dataset()
    df_raw = load_raw()
except FileNotFoundError:
    print('Raw file not found — generating synthetic seed instead.')
    from src.data_loader import _make_seed_df
    df_raw = _make_seed_df()

df_raw.head()

In [ ]:
eda_summary(df_raw)

## 2. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw['unit_price'].clip(upper=df_raw['unit_price'].quantile(0.99)),
             bins=50, color='#2E75B6', edgecolor='white')
axes[0].set_title('Unit Price Distribution (clipped at 99th pct)')
axes[0].set_xlabel('Price (₹)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df_raw['unit_price']), bins=50, color='#70AD47', edgecolor='white')
axes[1].set_title('log(1 + Price) Distribution')
axes[1].set_xlabel('log Price')

plt.tight_layout()
plt.savefig('../reports/price_distribution.png', bbox_inches='tight')
plt.show()
print('Skewness:', round(df_raw["unit_price"].skew(), 3))

## 3. Demand Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_raw['units_sold'].clip(upper=df_raw['units_sold'].quantile(0.99)),
        bins=50, color='#C00000', edgecolor='white')
ax.set_title('Units Sold Distribution (clipped at 99th pct)')
ax.set_xlabel('Units Sold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Price by Category

In [ ]:
if 'product_category' in df_raw.columns:
    top_cats = df_raw['product_category'].value_counts().head(10).index
    fig, ax = plt.subplots(figsize=(10, 5))
    df_raw[df_raw['product_category'].isin(top_cats)].boxplot(
        column='unit_price', by='product_category', ax=ax,
        flierprops=dict(marker='.', alpha=0.3)
    )
    ax.set_title('Unit Price by Top-10 Product Categories')
    ax.set_xlabel('Category')
    ax.set_ylabel('Price (₹)')
    plt.suptitle('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../reports/price_by_category.png', bbox_inches='tight')
    plt.show()
else:
    print('No product_category column — skipping.')

## 5. Correlation Heatmap

In [ ]:
num_df = df_raw.select_dtypes(include='number')
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix — Raw Features')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 6. Build Synthetic 500K Dataset
> We augment the seed data to 500K rows via bootstrap + Gaussian noise to match the scale referenced in the project description.

In [ ]:
df_synth = build_synthetic_dataset(df_raw, target_rows=500_000)
print(f'Synthetic dataset: {df_synth.shape[0]:,} rows × {df_synth.shape[1]} cols')
df_synth.head(3)

## 7. Seasonality Check

In [ ]:
df_synth['month'] = pd.to_datetime(df_synth['date']).dt.month
monthly = df_synth.groupby('month').agg(
    avg_price=('unit_price', 'mean'),
    avg_demand=('units_sold', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(monthly['month'], monthly['avg_price'], color='#2E75B6')
axes[0].set_title('Average Price by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Avg Price (₹)')
axes[0].set_xticks(range(1, 13))

axes[1].bar(monthly['month'], monthly['avg_demand'], color='#70AD47')
axes[1].set_title('Average Demand by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Avg Units Sold')
axes[1].set_xticks(range(1, 13))

plt.tight_layout()
plt.savefig('../reports/seasonality.png', bbox_inches='tight')
plt.show()

## Key Takeaways
- Price distributions are **right-skewed** → log transformation is well-motivated
- Clear **seasonality signal** in monthly demand — Dec peak, Jun trough
- Price–demand **negative correlation** across categories (as expected)
- 500K synthetic rows preserve seed distribution — bootstrapping is valid

➡ Proceed to `02_feature_engineering.ipynb`